# Import

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi

In [2]:
spark = SparkSession.builder \
    .appName("Agregasi-Sumarisasi") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark

# Proses

In [3]:
df = spark.read.parquet(
    "hdfs://namenode:9000/data/processed/fintech/fraudTrain_clean")
df.printSchema()


root
 |-- cc_num: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- age: long (nullable = true)
 |-- distance: double (nullabl

## Aggregation & Summarization

## Summary by Category

In [4]:
summary_by_category = (
    df.groupBy("category")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count"),
    )
    .orderBy(desc("fraud_count"))
)

summary_by_category.show()

+--------------+-----------------+--------------------+------------------+-----------+
|      category|total_transaction|        total_amount|        avg_amount|fraud_count|
+--------------+-----------------+--------------------+------------------+-----------+
|   grocery_pos|           123638|1.4460822380000055E7|116.96098594283356|       1743|
|  shopping_net|            97543|   8625149.680000028|  88.4240763560689|       1713|
|      misc_net|            63287|   5117709.260000003| 80.86509488520554|        915|
|  shopping_pos|           116672|          9307993.61| 79.77915532432803|        843|
| gas_transport|           131659|   8351732.290000028|  63.4345718105107|        618|
|      misc_pos|            79655|   5009582.500000003| 62.89099868181536|        250|
|     kids_pets|           113035|   6503680.160000006|57.536870526828025|        239|
| entertainment|            94014|   6036678.560000006|  64.2104214265961|        233|
| personal_care|            90758|   435345

## Spark SQL - Summary by Category

In [5]:
df.createOrReplaceTempView("fraud")

sql_fraud_category = spark.sql("""
SELECT
    category,
    COUNT(*) AS total_transaction,
    SUM(amt) AS total_amount,
    AVG(amt) AS avg_amount,
    SUM(is_fraud) AS fraud_count
    
FROM fraud
GROUP BY category
ORDER BY fraud_count DESC
""")

sql_fraud_category.show()

+--------------+-----------------+--------------------+------------------+-----------+
|      category|total_transaction|        total_amount|        avg_amount|fraud_count|
+--------------+-----------------+--------------------+------------------+-----------+
|   grocery_pos|           123638|1.4460822380000055E7|116.96098594283356|       1743|
|  shopping_net|            97543|   8625149.680000028|  88.4240763560689|       1713|
|      misc_net|            63287|   5117709.260000003| 80.86509488520554|        915|
|  shopping_pos|           116672|          9307993.61| 79.77915532432803|        843|
| gas_transport|           131659|   8351732.290000028|  63.4345718105107|        618|
|      misc_pos|            79655|   5009582.500000003| 62.89099868181536|        250|
|     kids_pets|           113035|   6503680.160000006|57.536870526828025|        239|
| entertainment|            94014|   6036678.560000006|  64.2104214265961|        233|
| personal_care|            90758|   435345

## Summary by Age Group

In [6]:
df = df.withColumn(
    "age_group",
    when(col("age") < 25, "18-24")
    .when(col("age") < 35, "25-34")
    .when(col("age") < 45, "35-44")
    .when(col("age") < 55, "45-54")
    .otherwise("55+")
)
summary_by_age = (
    df.groupBy("age_group")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count"),
    )
    .orderBy("age_group")
)

summary_by_age.show()


+---------+-----------------+--------------------+-----------------+-----------+
|age_group|total_transaction|        total_amount|       avg_amount|fraud_count|
+---------+-----------------+--------------------+-----------------+-----------+
|    18-24|           121663|   7525205.610000019|61.85286907276673|        764|
|    25-34|           287680| 2.146059773999993E7|74.59885198832012|       1391|
|    35-44|           272952|2.0683364669999983E7|75.77656390134523|       1163|
|    45-54|           256600|1.7702977710000012E7|68.99056005455968|       1491|
|      55+|           357780| 2.385028317000007E7| 66.6618681033039|       2697|
+---------+-----------------+--------------------+-----------------+-----------+



## Summary by Distance Group

In [7]:
quantiles = df.approxQuantile("distance", [0.33, 0.66], 0)
q1, q2 = quantiles
print(f"Kuantil 1: {q1}, Kuantil 2: {q2}")

df = df.withColumn(
    "distance_group",
     when(col("distance") < q1, "Near")
    .when(col("distance") < q2, "Medium")
    .otherwise("Far")
)

summary_by_distance = (
    df.groupBy("distance_group")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count")
    )
)

summary_by_distance.show()

Kuantil 1: 0.6491159993722272, Kuantil 2: 0.9169204323849363
+--------------+-----------------+--------------------+-----------------+-----------+
|distance_group|total_transaction|        total_amount|       avg_amount|fraud_count|
+--------------+-----------------+--------------------+-----------------+-----------+
|           Far|           440870| 3.096381424000006E7|70.23343443645533|       2554|
|          Near|           427902|3.0275232669999797E7|70.75272532028315|       2479|
|        Medium|           427903|2.9983381989999667E7|70.07051128409866|       2473|
+--------------+-----------------+--------------------+-----------------+-----------+



## Summary by Amount Group

In [8]:
quantiles = df.approxQuantile("amt",[0.33, 0.66],0)
q1, q2 = quantiles
print(f"Kuantil 1: {q1}, Kuantil 2: {q2}")


df = df.withColumn(
    "amount_group",
    when(col("amt") < q1, "Low")
    .when(col("amt") < q2, "Medium")
    .otherwise("High")
)

df = df.withColumn(
    "amount_group",
    when(col("amt") < 50, "Low")
    .when(col("amt") < 200, "Medium")
    .otherwise("High")
)

summary_by_amount = (
    df.groupBy("amount_group")
    .agg(
        count("*").alias("total_transaction"),
        sum("is_fraud").alias("fraud_count")
    )
    .orderBy("amount_group") 
)

summary_by_amount.show()

Kuantil 1: 20.43, Kuantil 2: 69.57
+------------+-----------------+-----------+
|amount_group|total_transaction|fraud_count|
+------------+-----------------+-----------+
|        High|            61930|       5704|
|         Low|           672214|       1607|
|      Medium|           562531|        195|
+------------+-----------------+-----------+



## Summary by Credit Card Number

In [9]:
summary_by_ccnum = (
    df.groupBy("cc_num","first","last","job")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count"),
    )
    .orderBy(desc("fraud_count"))
)

summary_by_ccnum.show()

+-------------------+--------+----------+--------------------+-----------------+------------------+------------------+-----------+
|             cc_num|   first|      last|                 job|total_transaction|      total_amount|        avg_amount|fraud_count|
+-------------------+--------+----------+--------------------+-----------------+------------------+------------------+-----------+
|   3520550088202337| Micheal|   Walters|   Freight forwarder|              989| 79783.68999999999| 80.67107178968654|         19|
|      4593569795412| Chelsea|     Silva|        Set designer|               19| 8431.929999999998|443.78578947368413|         19|
|      4260128500325| Whitney| Gallagher|Conservation offi...|             1466|108067.99000000003| 73.71622783083222|         18|
|   3575540972310993|  Rachel|Villarreal|             Curator|             1542| 91702.69000000003| 59.46996757457849|         16|
|   2720433095629877|    Mark|      Wood|Engineer, electro...|             3107|164

## Window Functions - Summary by Merchant

In [10]:
merchant_risk = (
    df.groupBy("merchant")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraud_count") /
            col("total_transaction") * 100, 2
        )
    )
    .filter(col("total_transaction") > 100)
)

window_spec = Window.orderBy(desc("fraud_count"))

summary_by_merchant = (
    merchant_risk
    .withColumn(
        "rank",
        dense_rank().over(window_spec)
    )
)

summary_by_merchant.show(10)

+--------------------+-----------------+------------------+------------------+-----------+----------+----+
|            merchant|total_transaction|      total_amount|        avg_amount|fraud_count|fraud_rate|rank|
+--------------------+-----------------+------------------+------------------+-----------+----------+----+
|  fraud_Rau and Sons|             2490|298354.76999999996|119.82119277108433|         49|      1.97|   1|
|   fraud_Cormier LLC|             3649|265129.91999999987| 72.65824061386678|         48|      1.32|   2|
|   fraud_Kozey-Boehm|             1866|183312.29000000004| 98.23809753483388|         48|      2.57|   2|
|   fraud_Kilback LLC|             4403|391078.15000000014| 88.82083806495574|         47|      1.07|   3|
|     fraud_Doyle Ltd|             2558| 300971.3700000001|117.65886239249419|         47|      1.84|   3|
|fraud_Vandervort-...|             2474|         292542.99|118.24696443007275|         47|       1.9|   3|
|      fraud_Kuhn LLC|             35

# Output

In [11]:
summary_by_category.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_category")
summary_by_age.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_age")
summary_by_distance.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_distance")
summary_by_amount.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_amount")
summary_by_ccnum.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_ccnum")
summary_by_merchant.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/summary_by_merchant")